# 1 Memory概述

Memory，是LangChain中用于多轮对话中保存和管理上下文信息（比如文本、图像、音频等）的组件。它让应用能够记住用户之前说了什么，从而实现对话的上下文感知能力，为构建真正智能和上下文感知的链式对话系统提供了基础。

Memory的设计理念

1. 输入问题：({"question": ...})
2. 读取历史消息：从Memory中READ历史消息（{"past_messages": [...]}）
3. 构建提示（Prompt)：读取到的历史消息和当前问题会被合并，构建一个新的Prompt
4. 模型处理：构建好的提示会被传递给语言模型进行处理。语言模型根据提示生成一个输出。
5. 解析输出：输出解析器通过正则表达式 regex("Answer: (.*)")来解析，返回一个回答（{"answer":
...}）给用户
6. 得到回复并写入Memory：新生成的回答会与当前的问题一起写入Memory，更新对话历史。
Memory会存储最新的对话内容，为后续的对话提供上下文支持。

不使用Memory模块，如何拥有记忆？

通过messages 变量，不断地将历史的对话信息追加到对话列表中，以此让大模型具备上下文记忆能力。

In [2]:
import os
import dotenv
from langchain_openai import ChatOpenAI
dotenv.load_dotenv()
os.environ['OPENAI_API_KEY'] = os.getenv("OPENAI_API_KEY")
os.environ['OPENAI_BASE_URL'] = os.getenv("OPENAI_BASE_URL")
# 创建大模型实例
llm = ChatOpenAI(model="gpt-4o-mini")



from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

def chat_with_model(question):
    # 步骤一：初始化消息
    chat_prompt_template = ChatPromptTemplate.from_messages([
        SystemMessage(content="你是一个有帮助的助手。"),
        HumanMessage(content="{question}")
    ])

    # 步骤二：定义一个循环体
    while True:
        # 步骤三：调用模型
        chain = chat_prompt_template | llm
        response = chain.invoke({"question": question})

        # 步骤四：输出模型的回答
        print("模型回答:", response.content)

        # 询问用户是否继续提问
        user_input = input("是否继续提问？(输入'退出'结束对话)")

        # 设置结束循环的条件
        if user_input.lower() == '退出':
            print("对话结束。")
            break

        #步骤五：记录用户回答
        chat_prompt_template.messages.append(AIMessage(content=response.content))
        chat_prompt_template.messages.append(HumanMessage(content=user_input))

chat_with_model()

模型回答: 你好！请问有什么我可以帮助你解决的问题呢？
模型回答: 你好！有什么我可以帮助你的吗？
模型回答: 李白是中国唐代著名的诗人，他的名字中的“白”并不是指颜色，而是用作姓氏。因此，李白并不以肤色为特征。李白以其豪放的个性和卓越的诗才而闻名于世。如果你对李白的诗歌或生平有兴趣，可以随时问我！
模型回答: 李白有许多著名的诗篇，以下是一些广为流传的作品：

1. **《将进酒》** - 这首诗表达了李白的豪情与对人生短暂的感悟，是他最为人熟知的作品之一。
  
2. **《庐山谣》** - 描绘了庐山的壮丽景色及其带来的自然哲思。

3. **《月下独酌》** - 描绘了李白在月下饮酒的情景，表现出孤独与对友人的怀念。

4. **《早发白帝城》** - 描述了在白帝城清晨出发的景象，风景如画，情感深邃。

5. **《蜀道难》** - 通过对蜀道艰难险阻的描写，表现了诗人的壮志与豪情。

6. **《静夜思》** - 描述了在寂静的夜晚，诗人思念家乡的情感，是李白极具代表性的一首诗。

这些诗篇展示了李白的才华与个性，不仅在中国文学史上有重要地位，也被广泛传唱。如果你需要更详细的分析或其它作品，可以告诉我！
模型回答: 刚才你聊的诗人是李白，他是中国唐代著名的诗人，以其豪放的个性和卓越的诗才而闻名。如果你还有其他问题或者想了解更多关于李白的诗歌或生平的内容，欢迎继续提问！
模型回答: 你刚才提到的诗人是李白。他是唐代最著名的诗人之一，以其豪放的个性和出色的诗才而闻名。如果你有更多问题或想了解关于李白的具体作品与风格，欢迎随时询问！
对话结束。


这种形式是最简单的一种让大模型具备上下文知识的存储方式，任何记忆的基础都是所有聊天交互的历史记录。即使这些不全部直接使用，也需要以某种形式存储。

# 2 基础Memory模块的使用

## 2.1 Memory模块的设计思路

如何设计Memory模块？

层次1(最直接的方式)：保留一个聊天消息列表

层次2(简单的新思路)：只返回最近交互的k条消息

层次3(稍微复杂一点)：返回过去k条消息的简洁摘要

层次4(更复杂)：从存储的消息中提取实体，并且仅返回有关当前运行中引用的实体的信息

## 2.2 ChatMessageHistory(基础)

ChatMessageHistory是一个用于存储和管理对话消息的基础类，它直接操作消息对象（如HumanMessage, AIMessage 等），是其它记忆组件的底层存储工具。

在API文档中，ChatMessageHistory 还有一个别名类：InMemoryChatMessageHistory；导包时，需使用：from langchain_classic.memory import ChatMessageHistory

from langchain_core.chat_history import InMemoryChatMessageHistory

场景1 记忆存储

In [10]:
#1.导入相关包
#from langchain_classic.memory import ChatMessageHistory
from langchain_core.chat_history import InMemoryChatMessageHistory

#2.实例化ChatMessageHistory对象
history = InMemoryChatMessageHistory()
# 3.添加UserMessage
history.add_user_message("hi!")

# 4.添加AIMessage
history.add_ai_message("whats up?")

# 5.返回存储的所有消息列表
print(history.messages)

[HumanMessage(content='hi!', additional_kwargs={}, response_metadata={}), AIMessage(content='whats up?', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]


场景2：对接LLM

In [4]:
from langchain_classic.memory import ChatMessageHistory
history = ChatMessageHistory()
history.add_ai_message("我是一个无所不能的小智")
history.add_user_message("你好，我叫小明，请介绍一下你自己")
history.add_user_message("我是谁呢？")

print(history.messages) #返回List[BaseMessage]类型

[AIMessage(content='我是一个无所不能的小智', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='你好，我叫小明，请介绍一下你自己', additional_kwargs={}, response_metadata={}), HumanMessage(content='我是谁呢？', additional_kwargs={}, response_metadata={})]


In [5]:
# 创建LLM
llm = ChatOpenAI(model_name='gpt-4o-mini')
llm.invoke(history.messages)

AIMessage(content='你好，小明！我是一种人工智能助手，旨在提供信息、回答问题以及帮助你解决各种问题。我可以与人对话，提供知识和建议。如果你有什么想知道的或者需要帮助的，随时可以问我！', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 53, 'prompt_tokens': 36, 'total_tokens': 89, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_f97eff32c5', 'id': 'chatcmpl-D8d1kFAnh0u2WN3ZkKVXmn6qkfW7h', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019c54c9-7c73-7ae1-a7a8-b11dec37fe9c-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 36, 'output_tokens': 53, 'total_tokens': 89, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

## 2.3 ConversationBufferMemory

ConversationBufferMemory是一个基础的对话记忆（Memory）组件，专门用于按原始顺序存储完整的对话历史。

适用场景：对话轮次较少、依赖完整上下文的场景（如简单的聊天机器）

场景1：入门使用

In [11]:
# 1.导入相关包
from langchain_classic.memory import ConversationBufferMemory

# 2.实例化ConversationBufferMemory对象
memory = ConversationBufferMemory()

# 3.保存消息到内存中
memory.save_context(inputs = {"input": "你好，我是人类"}, outputs = {"output": "你好，我是AI助手"})
memory.save_context(inputs = {"input": "很开心认识你"}, outputs = {"output": "我也是"})

# 4.读取内存中消息
print(memory.load_memory_variables({}))

#说明：返回的字典结构的key叫history，value是一个字符串，包含了之前保存的对话内容。

{'history': 'Human: 你好，我是人类\nAI: 你好，我是AI助手\nHuman: 很开心认识你\nAI: 我也是'}


举例2 以消息列表的方式返回存储的信息

In [13]:
# 1.导入相关包
from langchain_classic.memory import ConversationBufferMemory

# 2.实例化ConversationBufferMemory对象
memory = ConversationBufferMemory(return_messages=True)

# 3.保存消息到内存中
memory.save_context(inputs = {"input": "你好，我是人类"}, outputs = {"output": "你好，我是AI助手"})
memory.save_context(inputs = {"input": "很开心认识你"}, outputs = {"output": "我也是"})

# 4.读取内存中消息
#返回消息列表的方式1
print(memory.load_memory_variables({}))


print("\n")

#返回消息列表的方式1
print(memory.chat_memory.messages)

#说明：返回的字典结构的key叫history，value是一个字符串，包含了之前保存的对话内容。

{'history': [HumanMessage(content='你好，我是人类', additional_kwargs={}, response_metadata={}), AIMessage(content='你好，我是AI助手', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='很开心认识你', additional_kwargs={}, response_metadata={}), AIMessage(content='我也是', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]}


[HumanMessage(content='你好，我是人类', additional_kwargs={}, response_metadata={}), AIMessage(content='你好，我是AI助手', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='很开心认识你', additional_kwargs={}, response_metadata={}), AIMessage(content='我也是', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]


举例3 结合大模型和提示词模板的使用（PromptTemplate）

In [16]:
# 1、创建大模型实例
llm = ChatOpenAI(model="gpt-4o-mini")

#2、提供提示词模板
from langchain_core.prompts import PromptTemplate
prompt = PromptTemplate.from_template(
    template="""你可以与人类对话，当前对话历史：{history}，
    人类问题：{question}，

    回复："""
)

# 3、创建Memory对象
from langchain_classic.memory import ConversationBufferMemory
memory = ConversationBufferMemory()

# 4、提供chain
#2_chains = prompt | memory | llm

from langchain_classic.chains import LLMChain
chain = LLMChain(prompt=prompt, llm=llm, memory=memory)

response = chain.invoke({"question": "你是谁？我叫小明"})
print(response)

{'question': '你是谁？我叫小明', 'history': '', 'text': '你好，小明！我是一款人工智能助手，有什么我可以帮助你的吗？'}


In [17]:
response = chain.invoke({"question": "我叫什么名字呢？"})
print(response)

{'question': '我叫什么名字呢？', 'history': 'Human: 你是谁？我叫小明\nAI: 你好，小明！我是一款人工智能助手，有什么我可以帮助你的吗？', 'text': '你叫小明！很高兴认识你，有什么我可以帮助你的吗？'}


举例4 基于举例3显式的设置memory的key的值

In [ ]:
# 1、创建大模型实例
llm = ChatOpenAI(model="gpt-4o-mini")

#2、提供提示词模板
from langchain_core.prompts import PromptTemplate
prompt = PromptTemplate.from_template(
    template="""你可以与人类对话，当前对话历史：{chat_history}，
    人类问题：{question}，

    回复："""
)

# 3、创建Memory对象
from langchain_classic.memory import ConversationBufferMemory
memory = ConversationBufferMemory(memory_key="chat_history")

# 4、提供chain
from langchain_classic.chains import LLMChain
chain = LLMChain(prompt=prompt, llm=llm, memory=memory)

response = chain.invoke({"question": "你是谁？我叫小明"})
print(response)

{'question': '你是谁？我叫小明', 'chat_history': '', 'text': '你好，小明！我是一个人工智能助手，很高兴与你对话。有什么我可以帮助你的吗？'}


不管inputs、outputs的key用什么名字，都认为inputs的key是human，outputs的key是AI。

打印的结果的json数据的key，默认是“history”。可以通过ConversationBufferMemory的memory_key 属性修改。

举例5 结合大模型和提示词模板的使用（ChatPromptTemplate）

In [19]:
# 1.导入相关包
from langchain_core.messages import SystemMessage
from langchain_classic.chains.llm import LLMChain
from langchain_classic.memory import ConversationBufferMemory
from langchain_core.prompts import MessagesPlaceholder,ChatPromptTemplate,HumanMessagePromptTemplate
from langchain_openai import ChatOpenAI

# 2.创建LLM
llm = ChatOpenAI(model_name='gpt-4o-mini')

# 3.创建Prompt
prompt = ChatPromptTemplate.from_messages([
    ("system","你是一个与人类对话的机器人。"),
    MessagesPlaceholder(variable_name='history'),
    ("human","问题：{question}")
])

# 4.创建Memory
memory = ConversationBufferMemory(return_messages=True)

# 5.创建LLMChain
llm_chain = LLMChain(prompt=prompt,llm=llm, memory=memory)

# 6.调用LLMChain
res1 = llm_chain.invoke({"question": "中国首都在哪里？"})
print(res1,end="\n\n")

{'question': '中国首都在哪里？', 'history': [HumanMessage(content='中国首都在哪里？', additional_kwargs={}, response_metadata={}), AIMessage(content='中国的首都在北京。', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])], 'text': '中国的首都在北京。'}



In [20]:
res2 = llm_chain.invoke({"question": "我刚刚问了什么问题？"})
print(res2,end="\n\n")

{'question': '我刚刚问了什么问题？', 'history': [HumanMessage(content='中国首都在哪里？', additional_kwargs={}, response_metadata={}), AIMessage(content='中国的首都在北京。', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='我刚刚问了什么问题？', additional_kwargs={}, response_metadata={}), AIMessage(content='你刚刚问了“中国首都在哪里？”这个问题。', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])], 'text': '你刚刚问了“中国首都在哪里？”这个问题。'}



## 2.4 ConversationChain

ConversationChain实际上是就是对ConversationBufferMemory 和LLMChain 进行了封装，并且提供一个默认格式的提示词模版（我们也可以不用），从而简化了初始化ConversationBufferMemory的步骤。


举例1：使用PromptTemplate

In [ ]:
# 1、创建大模型实例
llm = ChatOpenAI(model="gpt-4o-mini")

#2、提供提示词模板
from langchain_core.prompts import PromptTemplate
prompt = PromptTemplate.from_template(
    template="""你可以与人类对话，当前对话历史：{history}，
    人类问题：{input}，

    回复："""
)

# # 3、创建Memory对象
# from langchain_classic.memory import ConversationBufferMemory
# memory = ConversationBufferMemory()

# # 4、提供chain
# from langchain_classic.chains import LLMChain
# 2_chains = LLMChain(prompt=prompt, llm=llm, memory=memory)


# 3、创建ConversationChain实例
from langchain_classic.chains import ConversationChain
chain = ConversationChain(llm=llm, prompt=prompt)

response = chain.invoke({"input": "你是谁？我叫小明"})
print(response)

{'input': '你是谁？我叫小明', 'history': '', 'response': '你好，小明！我是一个人工智能助手，很高兴和你聊天。你有什么想聊的吗？'}


In [24]:
response = chain.invoke({"input": "我叫什么名字呢？"})
print(response)

{'input': '我叫什么名字呢？', 'history': 'Human: 你是谁？我叫小明\nAI: 你好，小明！我是一个人工智能助手，很高兴和你聊天。你有什么想聊的吗？', 'response': '你叫小明！很高兴认识你。你今天过得怎么样？有什么想聊的呢？'}


举例2 使用内置默认格式的提示词模版（内部包含input、history变量）

In [ ]:
# 1、创建大模型实例
llm = ChatOpenAI(model="gpt-4o-mini")

#2、提供提示词模板
# from langchain_core.prompts import PromptTemplate
# prompt = PromptTemplate.from_template(
#     template="""你可以与人类对话，当前对话历史：{history}，
#     人类问题：{input}，

#     回复："""
# )

# # 3、创建Memory对象
# from langchain_classic.memory import ConversationBufferMemory
# memory = ConversationBufferMemory()

# # 4、提供chain
# from langchain_classic.chains import LLMChain
# 2_chains = LLMChain(prompt=prompt, llm=llm, memory=memory)


# 3、创建ConversationChain实例（内部提供了默认的提示词模板，且此模板的变量是input、history
from langchain_classic.chains import ConversationChain
chain = ConversationChain(llm=llm)

response = chain.invoke({"input": "你是谁？我叫小明"})
print(response)

{'input': '你是谁？我叫小明', 'history': '', 'response': '你好，小明！我是一种人工智能助手，旨在帮助你回答问题、提供信息和进行有趣的对话。我可以和你讨论很多主题，比如科学、历史、娱乐、文学等等。你今天想聊些什么呢？'}


In [26]:
response = chain.invoke({"input": "我叫什么名字呢？"})
print(response)

{'input': '我叫什么名字呢？', 'history': 'Human: 你是谁？我叫小明\nAI: 你好，小明！我是一种人工智能助手，旨在帮助你回答问题、提供信息和进行有趣的对话。我可以和你讨论很多主题，比如科学、历史、娱乐、文学等等。你今天想聊些什么呢？', 'response': '你的名字是小明！很高兴认识你！如果你想聊聊你的兴趣或者有其他问题，随时告诉我哦！'}


举例3 

In [27]:
# 1.导入所需的库
from langchain_openai import ChatOpenAI
from langchain_classic.chains import ConversationChain
# 2.初始化大语言模型
llm = ChatOpenAI(model="gpt-4o-mini")
# 3.初始化对话链
conv_chain = ConversationChain(llm=llm)
# 4.进行对话
resut1 = conv_chain.invoke(input="小明有1只猫")
# print(resut1)
resut2 = conv_chain.invoke(input="小刚有2只狗")
# print(resut2)
resut3 = conv_chain.invoke(input="小明和小刚一共有几只宠物?")
print(resut3)

{'input': '小明和小刚一共有几只宠物?', 'history': 'Human: 小明有1只猫\nAI: 小明有1只猫真不错！猫咪常常是很好的陪伴，它们通常性格独立，但又会在你身边撒娇。你知道小明的猫是什么品种的吗？还有，它的名字是什么？猫咪的饮食和护理也很重要哦，比如每天需要合适的食物和定期的兽医检查。希望小明的猫咪健康快乐！\nHuman: 小刚有2只狗\nAI: 小刚有2只狗真令人羡慕！狗狗通常是非常忠诚和活泼的宠物，它们不仅可以陪伴我们，还能带给我们很多快乐和欢笑。你知道小刚的狗是什么品种吗？狗的品种有很多，比如金毛寻回犬、拉布拉多、柴犬等等，不同品种的狗有不同的性格和需求。还有，它们的名字是什么呢？狗狗通常需要定期的锻炼，比如散步或玩耍，以保持健康和快乐。同时，适当的饮食和兽医检查也是必不可少的。希望小刚的狗狗们都能健康快乐！', 'response': '小明有1只猫，小刚有2只狗，所以他们一共拥有3只宠物！这样的小团队一定非常有趣，猫咪和狗狗在一起可能会有很多有趣的互动。如果你有时间，可以想象一下它们一起玩耍的样子！你觉得小明的猫和小刚的狗会成为好朋友吗？'}


## 2.5 ConversationBufferWindowMemory

在了解了ConversationBufferMemory记忆类后，我们知道了它能够无限的将历史对话信息填充到
History中，从而给大模型提供上下文的背景。但这会导致内存量十分大，并且消耗的token是非常多的，此外，每个大模型都存在最大输入的Token限制。

过久远的对话数据往往并不能对当前轮次的问答提供有效的信息，LangChain 给出的解决方
式是： ConversationBufferWindowMemory 模块。该记忆类会保存一段时间内对话交互的列表， 仅使用最近 K 个交互。这样就使缓存区不会变得太大。

特点：

适合长对话场景。

与 Chains/Models 无缝集成

支持两种返回格式（通过 return_messages 参数控制输出格式）

    return_messages=True 返回消息对象列表（List[BaseMessage]）
    
    return_messages=False （默认） 返回拼接的纯文本字符串

举例1 

In [29]:
# 1.导入相关包
from langchain_classic.memory import ConversationBufferWindowMemory

# 2.实例化ConversationBufferWindowMemory对象，设定窗口阈值
memory = ConversationBufferWindowMemory(k=2)

# 3.保存消息
memory.save_context({"input": "你好"}, {"output": "怎么了"})
memory.save_context({"input": "你是谁"}, {"output": "我是AI助手"})
memory.save_context({"input": "你的生日是哪天？"}, {"output": "我不清楚"})

# 4.读取内存中消息（返回消息内容的纯文本）
print(memory.load_memory_variables({}))

{'history': 'Human: 你是谁\nAI: 我是AI助手\nHuman: 你的生日是哪天？\nAI: 我不清楚'}


举例2 返回消息构成的上下文记忆

In [30]:
# 1.导入相关包
from langchain_classic.memory import ConversationBufferWindowMemory

# 2.实例化ConversationBufferWindowMemory对象，设定窗口阈值
memory = ConversationBufferWindowMemory(k=2, return_messages=True)

# 3.保存消息
memory.save_context({"input": "你好"}, {"output": "怎么了"})
memory.save_context({"input": "你是谁"}, {"output": "我是AI助手"})
memory.save_context({"input": "你的生日是哪天？"}, {"output": "我不清楚"})

# 4.读取内存中消息（返回消息内容的纯文本）
print(memory.load_memory_variables({}))

{'history': [HumanMessage(content='你是谁', additional_kwargs={}, response_metadata={}), AIMessage(content='我是AI助手', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='你的生日是哪天？', additional_kwargs={}, response_metadata={}), AIMessage(content='我不清楚', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]}


场景2：结合chain

借助提示词模版去构建LangChain

举例1 

In [32]:
from langchain_classic.memory import ConversationBufferWindowMemory
# 1.导入相关包
from langchain_core.prompts.prompt import PromptTemplate
from langchain_classic.chains.llm import LLMChain

# 2.定义模版
template = """以下是人类与AI之间的友好对话描述。AI表现得很健谈，并提供了大量来自其上下文的
具体细节。如果AI不知道问题的答案，它会表示不知道。

当前对话：
{history}
Human: {question}
AI:"""

# 3.定义提示词模版
prompt_template = PromptTemplate.from_template(template)

# 4.创建大模型
llm = ChatOpenAI(model="gpt-4o-mini")

# 5.实例化ConversationBufferWindowMemory对象，设定窗口阈值
memory = ConversationBufferWindowMemory(k=1)

# 6.定义LLMChain
conversation_with_summary = LLMChain(
    llm=llm,
    prompt=prompt_template,
    memory=memory,
    #verbose=True,
)

# 7.执行链（第一次提问）
respon1 = conversation_with_summary.invoke({"question":"你好，我是孙小空"})
# print(respon1)

# 8.执行链（第二次提问）
respon2 =conversation_with_summary.invoke({"question":"我还有两个师弟，一个是猪小戒，一个是沙小僧"})
# print(respon2)

# 9.执行链（第三次提问）
respon3 =conversation_with_summary.invoke({"question":"我今年高考，竟然考上了1本"})
# print(respon3)

# 10.执行链（第四次提问）
respon4 =conversation_with_summary.invoke({"question":"我叫什么？"})
print(respon4)

{'question': '我叫什么？', 'history': 'Human: 我今年高考，竟然考上了1本\nAI: 太棒了！恭喜你考上一本大学！这一定是你努力学习的结果。你打算选择什么专业呢？或者你对未来的大学生活有什么期待？', 'text': '抱歉，我不知道你叫什么名字。如果你愿意，可以告诉我你叫什么名字，我很乐意记住！你对大学生活还有什么其他的期待吗？'}


举例2 基于举例1，修改参数k=3

In [33]:
from langchain_classic.memory import ConversationBufferWindowMemory
# 1.导入相关包
from langchain_core.prompts.prompt import PromptTemplate
from langchain_classic.chains.llm import LLMChain

# 2.定义模版
template = """以下是人类与AI之间的友好对话描述。AI表现得很健谈，并提供了大量来自其上下文的
具体细节。如果AI不知道问题的答案，它会表示不知道。

当前对话：
{history}
Human: {question}
AI:"""

# 3.定义提示词模版
prompt_template = PromptTemplate.from_template(template)

# 4.创建大模型
llm = ChatOpenAI(model="gpt-4o-mini")

# 5.实例化ConversationBufferWindowMemory对象，设定窗口阈值
memory = ConversationBufferWindowMemory(k=3)

# 6.定义LLMChain
conversation_with_summary = LLMChain(
    llm=llm,
    prompt=prompt_template,
    memory=memory,
    #verbose=True,
)

# 7.执行链（第一次提问）
respon1 = conversation_with_summary.invoke({"question":"你好，我是孙小空"})
# print(respon1)

# 8.执行链（第二次提问）
respon2 =conversation_with_summary.invoke({"question":"我还有两个师弟，一个是猪小戒，一个是沙小僧"})
# print(respon2)

# 9.执行链（第三次提问）
respon3 =conversation_with_summary.invoke({"question":"我今年高考，竟然考上了1本"})
# print(respon3)

# 10.执行链（第四次提问）
respon4 =conversation_with_summary.invoke({"question":"我叫什么？"})
print(respon4)

{'question': '我叫什么？', 'history': 'Human: 你好，我是孙小空\nAI: 你好，孙小空！很高兴见到你。你今天过得怎么样？有什么想聊的话题吗？\nHuman: 我还有两个师弟，一个是猪小戒，一个是沙小僧\nAI: 很有趣的名字！听起来你们之间的关系很融洽。猪小戒和沙小僧有什么特别的故事或者有趣的经历吗？你们通常一起做些什么呢？\nHuman: 我今年高考，竟然考上了1本\nAI: 太棒了，恭喜你考上一本！这是一个很了不起的成就。你打算去哪所学校呢？或者你对未来的专业有什么想法吗？', 'text': '你叫孙小空！这是一个很有个性的名字。你喜欢这个名字吗？'}


# 3 其他Memory模块

## 3.1 ConversationTokenBufferMemory

ConversationTokenBufferMemory 是 LangChain 中一种基于Token 数量控制的对话记忆机制。如果字符数量超出指定数目，它会切掉这个对话的早期部分，以保留与最近的交流相对应的字符数量。

举例 情况1：

In [39]:
# 1.导入相关包
from langchain_classic.memory import ConversationTokenBufferMemory
from langchain_openai import ChatOpenAI

# 2.创建大模型
llm = ChatOpenAI(model="gpt-4o-mini")

# 3.定义ConversationTokenBufferMemory对象
memory = ConversationTokenBufferMemory(
    llm=llm,
    max_token_limit=20 # 设置token上限,默认值为2000
)

# 添加对话
memory.save_context({"input": "你好吗？"}, {"output": "我很好，谢谢！"})
memory.save_context({"input": "今天天气如何？"}, {"output": "晴天，25度"})

# 查看当前记忆
print(memory.load_memory_variables({}))

{'history': 'AI: 晴天，25度'}


## 3.2 ConversationSummaryMemory

前⾯的⽅式发现，如果全部保存下来太过浪费，截断时⽆论是按照对话条数还是token 都是⽆法保证既节省内存⼜保证对话质量的，所以推出ConversationSummaryMemory、
ConversationSummaryBufferMemory

ConversationSummaryMemory是 LangChain 中一种智能压缩对话历史的记忆机制，它通过大语言模型(LLM)自动生成对话内容的精简摘要，而不是存储原始对话文本。
这种记忆方式特别适合长对话和需要保留核心信息的场景。

场景1：

如果实例化ConversationSummaryMemory前，没有历史消息，可以使用构造方法实例化

In [41]:
# 1.导入相关包
from langchain_classic.memory import ConversationSummaryMemory, ChatMessageHistory
from langchain_openai import ChatOpenAI

# 2.创建大模型
llm = ChatOpenAI(model="gpt-4o-mini")

# 3.定义ConversationSummaryMemory对象
memory = ConversationSummaryMemory(llm=llm)

# 4.存储消息
memory.save_context({"input": "你好"}, {"output": "怎么了"})
memory.save_context({"input": "你是谁"}, {"output": "我是AI助手小智"})
memory.save_context({"input": "初次对话，你能介绍一下你自己吗？"}, {"output": "当然可以了。我是一个无所不能的小智。"})

# 5.读取消息（总结后的）
print(memory.load_memory_variables({}))

{'history': 'The human greets the AI with "你好" (hello). The AI responds by asking "怎么了" (what\'s wrong?). The human then asks, "你是谁" (who are you?), and the AI replies, "我是AI助手小智" (I am the AI assistant Xiao Zhi). The human requests an introduction, and the AI responds that it is an all-capable assistant named Xiao Zhi.'}


场景2：

如果实例化ConversationSummaryMemory前，已经有历史消息，可以调用from_messages()实例化

In [42]:
# 1.导入相关包
from langchain_classic.memory import ConversationSummaryMemory, ChatMessageHistory
from langchain_openai import ChatOpenAI

# 2.定义ChatMessageHistory对象
llm = ChatOpenAI(model="gpt-4o-mini")

# 3.假设原始消息
history = ChatMessageHistory()
history.add_user_message("你好，你是谁？")
history.add_ai_message("我是AI助手小智")

# 4.初始化ConversationSummaryMemory实例
memory = ConversationSummaryMemory.from_messages(
    llm=llm,
    #是生成摘要的原材料 保留完整对话供必要时回溯。当新增对话时，LLM需要结合原始历史生成新摘要
    chat_memory=history,
)

print(memory.load_memory_variables({}))

memory.save_context(inputs={"human":"我的名字叫小明"},outputs={"AI":"很高兴认识你"})

print(memory.load_memory_variables({}))

print(memory.chat_memory.messages)

{'history': 'The human greets the AI and asks who it is. The AI responds that it is the AI assistant named Xiao Zhi.'}
{'history': 'The human greets the AI and asks who it is. The AI responds that it is the AI assistant named Xiao Zhi. The human introduces himself as Xiao Ming, and the AI expresses pleasure in meeting him.'}
[HumanMessage(content='你好，你是谁？', additional_kwargs={}, response_metadata={}), AIMessage(content='我是AI助手小智', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='我的名字叫小明', additional_kwargs={}, response_metadata={}), AIMessage(content='很高兴认识你', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]


## 3.3 ConversationSummaryBufferMemory

ConversationSummaryBufferMemory 是 LangChain 中一种混合型记忆机制，它结合了ConversationBufferMemory（完整对话记录）和 ConversationSummaryMemory（摘要记忆）的优点，在保留最近对话原始记录的同时，对较早的对话内容进行智能摘要。

场景1：入门使用

情况1：构造方法实例化，并设置max_token_limit

In [45]:
# 1.导入相关的包
from langchain_classic.memory import ConversationSummaryBufferMemory
from langchain_openai import ChatOpenAI

# 2.定义模型
llm = ChatOpenAI(model_name="gpt-4o-mini",temperature=0)

# 3.定义ConversationSummaryBufferMemory对象
memory = ConversationSummaryBufferMemory(
    llm=llm, 
    max_token_limit=40, #控制缓缓存区的大小，默认值为2000
    return_messages=True
)

# 4.保存消息
memory.save_context({"input": "你好，我的名字叫小明"}, {"output": "很高兴认识你，小明"})
memory.save_context({"input": "李白是哪个朝代的诗人"}, {"output": "李白是唐朝诗人"})
memory.save_context({"input": "唐宋八大家里有苏轼吗？"}, {"output": "有"})

# 5.读取内容
print(memory.load_memory_variables({}))
print("\n")
print(memory.chat_memory.messages)

{'history': [SystemMessage(content='The human introduces themselves as Xiao Ming. The AI expresses pleasure in meeting Xiao Ming, and the human asks which dynasty the poet Li Bai belongs to.', additional_kwargs={}, response_metadata={}), AIMessage(content='李白是唐朝诗人', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='唐宋八大家里有苏轼吗？', additional_kwargs={}, response_metadata={}), AIMessage(content='有', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]}


[AIMessage(content='李白是唐朝诗人', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='唐宋八大家里有苏轼吗？', additional_kwargs={}, response_metadata={}), AIMessage(content='有', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]


对比组

In [46]:
# 1.导入相关的包
from langchain_classic.memory import ConversationSummaryBufferMemory
from langchain_openai import ChatOpenAI

# 2.定义模型
llm = ChatOpenAI(model_name="gpt-4o-mini",temperature=0)

# 3.定义ConversationSummaryBufferMemory对象
memory = ConversationSummaryBufferMemory(
    llm=llm, 
    max_token_limit=100, #控制缓缓存区的大小，默认值为2000
    return_messages=True
)

# 4.保存消息
memory.save_context({"input": "你好，我的名字叫小明"}, {"output": "很高兴认识你，小明"})
memory.save_context({"input": "李白是哪个朝代的诗人"}, {"output": "李白是唐朝诗人"})
memory.save_context({"input": "唐宋八大家里有苏轼吗？"}, {"output": "有"})

# 5.读取内容
print(memory.load_memory_variables({}))
print("\n")
print(memory.chat_memory.messages)

{'history': [HumanMessage(content='你好，我的名字叫小明', additional_kwargs={}, response_metadata={}), AIMessage(content='很高兴认识你，小明', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='李白是哪个朝代的诗人', additional_kwargs={}, response_metadata={}), AIMessage(content='李白是唐朝诗人', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='唐宋八大家里有苏轼吗？', additional_kwargs={}, response_metadata={}), AIMessage(content='有', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]}


[HumanMessage(content='你好，我的名字叫小明', additional_kwargs={}, response_metadata={}), AIMessage(content='很高兴认识你，小明', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='李白是哪个朝代的诗人', additional_kwargs={}, response_metadata={}), AIMessage(content='李白是唐朝诗人', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='唐宋

场景2：客服

In [47]:
from langchain_classic.memory import ConversationSummaryBufferMemory
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_classic.chains.llm import LLMChain

# 1、初始化大语言模型
llm = ChatOpenAI(
model="gpt-4o-mini",
temperature=0.5,
max_tokens=500
)

# 2、定义提示模板
prompt = ChatPromptTemplate.from_messages([
    ("system", "你是电商客服助手，用中文友好回复用户问题。保持专业但亲切的语气。"),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{input}")
])

# 3、创建带摘要缓冲的记忆系统
memory = ConversationSummaryBufferMemory(
    llm=llm,
    max_token_limit=400,
    memory_key="chat_history",
    return_messages=True
)

# 4、创建对话链
chain = LLMChain(
    llm=llm,
    prompt=prompt,
    memory=memory,
)

# 5、模拟多轮对话
dialogue = [
    ("你好，我想查询订单12345的状态", None),
    ("这个订单是上周五下的", None),
    ("我现在急着用，能加急处理吗", None),
    ("等等，我可能记错订单号了，应该是12346", None),
    ("对了，你们退货政策是怎样的", None)
]

# 6、执行对话
for user_input, _ in dialogue:
    response = chain.invoke({"input": user_input})
    print(f"用户: {user_input}")
    print(f"客服: {response['text']}\n")
    
# 7、查看当前记忆状态
print("\n=== 当前记忆内容 ===")
print(memory.load_memory_variables({}))

用户: 你好，我想查询订单12345的状态
客服: 您好！感谢您联系我们。关于订单12345的状态，我会为您查询一下。请稍等片刻。 

（如果系统可以查询到状态，您可以继续提供相关信息；如果不能查询，请告知用户我们会尽快处理。）

用户: 这个订单是上周五下的
客服: 感谢您提供订单信息！请您稍等片刻，我将为您查询订单12345的状态。 

（如果能查询到状态，请提供具体信息；如果不能查询，请告知用户我们会尽快处理。）

用户: 我现在急着用，能加急处理吗
客服: 我理解您急需这个订单的心情。请您放心，我会尽快为您处理。 

请您提供一下您的联系方式，我将尽快与相关部门沟通，看看是否可以加急处理您的订单。同时，您也可以查看一下我们的加急服务政策，看看是否符合条件。感谢您的理解与支持！

用户: 等等，我可能记错订单号了，应该是12346
客服: 没问题！感谢您更正订单号。请您稍等片刻，我将为您查询订单12346的状态。 

（如果能查询到状态，请提供具体信息；如果不能查询，请告知用户我们会尽快处理。）

用户: 对了，你们退货政策是怎样的
客服: 我们的退货政策如下：

1. **退货期限**：您可以在收到商品后7天内申请退货。
2. **商品状态**：退货的商品必须保持未使用状态，且原包装、标签完整。
3. **申请流程**：请您登录账户，进入“我的订单”，找到相关订单，选择“申请退货”，并按照提示填写相关信息。
4. **费用**：如果是由于商品质量问题导致的退货，我们将承担退货运费；如果是因为个人原因（如不喜欢、尺寸不合等），运费由您承担。

如果您还有其他问题，欢迎随时询问！我们会尽力为您提供帮助。


=== 当前记忆内容 ===
{'chat_history': [SystemMessage(content='The human inquires about the status of order 12345. The AI responds by thanking the human for contacting them and states that it will check the status of the order, asking the human to wait a moment.', additional_kwargs={}, re

## 3.4 ConversationEntityMemory(了解)

ConversationEntityMemory 是一种基于实体的对话记忆机制，它能够智能地识别、存储和利用对话中出现的实体信息（如人名、地点、产品等）及其属性/关系，并结构化存储，使 AI 具备更强的上下文理解和记忆能力。

好处：解决信息过载问题

应用场景：在医疗等高风险领域，必须用实体记忆确保关键信息（如过敏史）被100%准确识别和拦
截。

{"input": "我头痛，血压140/90，在吃阿司匹林。"},

{"output": "建议监测血压，阿司匹林可继续服用。"}

{"input": "我对青霉素过敏。"},

{"output": "已记录您的青霉素过敏史。"}

{"input": "阿司匹林吃了三天，头痛没缓解。"},

{"output": "建议停用阿司匹林，换布洛芬试试。"}

使用ConversationSummaryMemory

"患者主诉头痛和⾼⾎压（140/90），正在服⽤阿司匹林。患者对⻘霉素过敏。三天后头痛未缓
解，建议更换⽌痛药。"

使用ConversationEntityMemory

{

"症状": "头痛",

"⾎压": "140/90",

"当前⽤药": "阿司匹林（⽆效）",

"过敏药物": "⻘霉素"

}

对比：ConversationSummaryMemory 和 ConversationEntityMemory

ConversationSummaryMemory:

自然语言文本（ 一段话）

需大模型“读懂”摘要文本，
如果 AI 的注意力集中在“头痛”和“换药”
上，可能会忽略过敏提示（尤其是摘要较长时）

ConversationEntityMemory:

结构化字典（ 键值对）

无需依赖模型的“阅读理解能
力”，直接通过字段名（如过敏药物）查询

防错可靠性高（通过代码强制检查）


举例

In [49]:
from langchain_classic.chains.llm import LLMChain
from langchain_classic.memory import ConversationEntityMemory
from langchain_classic.memory.prompt import ENTITY_MEMORY_CONVERSATION_TEMPLATE
from langchain_openai import ChatOpenAI

# 初始化大语言模型
llm = ChatOpenAI(model_name='gpt-4o-mini', temperature=0)

# 使用LangChain为实体记忆设计的预定义模板
prompt = ENTITY_MEMORY_CONVERSATION_TEMPLATE

# 初始化实体记忆
memory = ConversationEntityMemory(llm=llm)

# 提供对话链
chain = LLMChain(
    llm=llm,
    prompt=prompt,
    memory=memory,
    #verbose=True, # 设置为True可以看到链的详细推理过程
)

# 进行几轮对话，记忆组件会在后台自动提取和存储实体信息
chain.invoke(input="你好，我叫蜘蛛侠。我的好朋友包括钢铁侠、美国队长和绿巨人。")
chain.invoke(input="我住在纽约。")
chain.invoke(input="我使用的装备是由斯塔克工业提供的。")

# 查询记忆体中存储的实体信息
print("\n当前存储的实体信息:")
print(chain.memory.entity_store.store)


当前存储的实体信息:
{'蜘蛛侠': '蜘蛛侠是一个超级英雄，他的好朋友包括钢铁侠、美国队长和绿巨人。', '钢铁侠': '钢铁侠是蜘蛛侠的好朋友之一。', '美国队长': '美国队长是蜘蛛侠的好朋友之一。', '绿巨人': '绿巨人是蜘蛛侠的好朋友之一。', '纽约': '蜘蛛侠住在纽约。', '斯塔克工业': '斯塔克工业提供了蜘蛛侠使用的装备。'}


In [50]:
# 基于记忆进行提问
answer = chain.invoke(input="你能告诉我蜘蛛侠住在哪里以及他的好朋友有哪些吗？")
print("\nAI的回答:")
print(answer)


AI的回答:
{'input': '你能告诉我蜘蛛侠住在哪里以及他的好朋友有哪些吗？', 'history': 'Human: 你好，我叫蜘蛛侠。我的好朋友包括钢铁侠、美国队长和绿巨人。\nAI: 你好，蜘蛛侠！很高兴认识你。你和钢铁侠、美国队长以及绿巨人都是超级英雄，真是一个强大的团队！你们最近有什么冒险吗？\nHuman: 我住在纽约。\nAI: 纽约是一个充满活力的城市，适合超级英雄们活动！你在纽约的生活怎么样？有没有遇到什么有趣的事情或者挑战？\nHuman: 我使用的装备是由斯塔克工业提供的。\nAI: 斯塔克工业的装备真是太棒了！钢铁侠的技术总是让人惊叹。你最喜欢哪一件装备？它在你的冒险中帮助了你哪些方面？', 'entities': {'蜘蛛侠': '蜘蛛侠是一个超级英雄，他的好朋友包括钢铁侠、美国队长和绿巨人。'}, 'text': '蜘蛛侠住在纽约市，他的好朋友包括钢铁侠、美国队长和绿巨人。这些超级英雄们常常一起合作，面对各种挑战和敌人。你对蜘蛛侠的故事有什么特别感兴趣的地方吗？'}


## 3.5 ConversationKGMemory(了解)

ConversationKGMemory是一种基于知识图谱（Knowledge Graph）的对话记忆模块，它比
ConversationEntityMemory 更进一步，不仅能识别和存储实体，还能捕捉实体之间的复杂关系，形成结构化的知识网络。

特点：

知识图谱结构 将对话内容转化为 (头实体, 关系, 尾实体) 的三元组形式

动态关系推理

pip install networkx

In [52]:
#1.导入相关包
from langchain_classic.memory import ConversationKGMemory
from langchain_openai import ChatOpenAI


# 2.定义LLM
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# 3.定义ConversationKGMemory对象
memory = ConversationKGMemory(llm=llm)

# 4.保存会话
memory.save_context({"input": "向山姆问好"}, {"output": "山姆是谁"})
memory.save_context({"input": "山姆是我的朋友"}, {"output": "好的"})

# 5.查询会话
memory.load_memory_variables({"input": "山姆是谁"})

{'history': 'On 山姆: 山姆 是 我的朋友.'}

添加信息

In [53]:
memory.get_knowledge_triplets("她最喜欢的颜色是红色")

[KnowledgeTriple(subject='山姆', predicate='是', object_='我的朋友'),
 KnowledgeTriple(subject='山姆', predicate='最喜欢的颜色', object_='红色')]

## 3.6 VectorStoreRetrieverMemory(了解)

VectorStoreRetrieverMemory是一种基于向量检索的先进记忆机制，它将对话历史存储在向量数据库中，通过语义相似度检索相关信息，而非传统的线性记忆方式。每次调用时，就会查找与该记忆关联最高的k个文档。

适用场景：这种记忆特别适合需要长期记忆和语义理解的复杂对话系统。

向量数据库能存储大量数据

举例：

In [57]:
import os
import dotenv
from langchain_openai import OpenAIEmbeddings

dotenv.load_dotenv()
os.environ['OPENAI_API_KEY'] = os.getenv("OPENAI_API_KEY")
os.environ['OPENAI_BASE_URL'] = os.getenv("OPENAI_BASE_URL")


# 1.导入相关包
from langchain_openai import OpenAIEmbeddings
from langchain_classic.memory import VectorStoreRetrieverMemory,ConversationBufferMemory
from langchain_community.vectorstores import FAISS

# 2.定义ConversationBufferMemory对象
memory = ConversationBufferMemory()
memory.save_context({"input": "我最喜欢的食物是披萨"}, {"output": "很高兴知道"})
memory.save_context({"Human": "我喜欢的运动是跑步"}, {"AI": "好的,我知道了"})
memory.save_context({"Human": "我最喜欢的运动是足球"}, {"AI": "好的,我知道了"})

# 3.定义向量嵌入模型
embeddings_model = OpenAIEmbeddings(
    model="text-embedding-ada-002"
)

# 4.初始化向量数据库
vectorstore = FAISS.from_texts(memory.buffer.split("\n"), embeddings_model) # 空初始化

# 5.定义检索对象
retriever = vectorstore.as_retriever(search_kwargs=dict(k=1))

# 6.初始化VectorStoreRetrieverMemory
memory = VectorStoreRetrieverMemory(retriever=retriever)
print(memory.load_memory_variables({"prompt": "我最喜欢的食物是"}))

{'history': 'Human: 我最喜欢的食物是披萨'}
